In [1]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

from google.colab import drive
drive.mount("/content/drive")

!pip -q install -U "ultralytics>=8.3.0" roboflow

True Tesla T4


MessageError: Error: credential propagation was unsuccessful

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="your-api-key")
project = rf.workspace("kaushik-gfstb").project("traffic-signs-pq69p")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Traffic-signs-1 in yolov8:: 100%|██████████| 16040/16040 [00:02<00:00, 5712.43it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=60,
    imgsz=320,
    batch=32,
    project="/content/drive/MyDrive/raasta/weights",
    name="m3_signs_v1",
)

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Traffic-signs-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=m3_signs_v1, nbs=64, nms=None, opset=None, optimize=Fal

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 56, 57, 58, 59, 60, 61, 62, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7897021f9160>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,

In [ ]:
best = "/content/drive/MyDrive/raasta/weights/m3_signs_v1/weights/best.pt"
metrics = YOLO(best).val(data=f"{dataset.location}/data.yaml", imgsz=320)
print("mAP50:", metrics.box.map50)

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,150,446 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1207.1±632.4 MB/s, size: 39.0 KB)
val: Scanning /content/Traffic-signs-1/valid/labels.cache... 720 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 720/720 251.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 6.5it/s 6.9s
                   all        720        811      0.839      0.752      0.818      0.576
                     -         98        171      0.822      0.677      0.804      0.415
- 50- probability of horizontal flip          6          6      0.816      0.667      0.948      0.643
- 50- probability of vertical flip          8         11          1      0.599      0.729      0.454
- Random Gaussian blur of between 0 and 3.25 pixels          8          8      0.965      0.875      0

In [ ]:
path = YOLO(best).export(format="tflite", imgsz=320)
print(path)
!cp "{path}" /content/drive/MyDrive/raasta/weights/raasta_signs.tflite

WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 73 layers, 3,150,446 parameters, 0 gradients, 2.2 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/raasta/weights/m3_signs_v1/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 78, 2100) (6.2 MB)
requirements: Ultralytics requirements ['litert-torch>=0.9.0', 'ai-edge-litert>=2.1.4'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 86 packages in 849ms
Prepared 16 packages in 4.53s
Uninstalled 4 packages in 236ms
Installed 16 packages in 192ms
 + ai-edge-litert==2.2.0
 + 

invalid escape sequence '\.'



LiteRT: starting export with litert_torch 0.9.4...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

...

(00:13) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:00)

(00:13) [ DONE] LiteRT-Torch Convert (+00:13)

(00:00) [START] Write Model to /content/drive/MyDrive/raasta/weights/m3_signs_v1/weights/best.tflite

(00:00) [ DONE] Write Model to /content/drive/MyDrive/raasta/weights/m3_signs_v1/weights/best.tflite (+00:00)

LiteRT: export success ✅ 26.7s, saved as '/content/drive/MyDrive/raasta/weights/m3_signs_v1/weights/best.tflite' (12.2 MB)

Export complete (26.8s)
Results saved to /content/drive/MyDrive/raasta/weights/m3_signs_v1/weights/best.tflite
Predict:         yolo predict task=detect model=/content/drive/MyDrive/raasta/weights/m3_signs_v1/weights/best.tflite imgsz=320 
Validate:        yolo val task=detect model=/content/drive/MyDrive/raasta/weights/m3_signs_v1/weights/best.tflite imgsz=320 data=/content/Traffic-signs-1/data.yaml  
Visualize:       https://netron.app
/content/drive/MyDrive/raasta/weights/m3_signs_v1/weights/best.tflite


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install -U "ultralytics>=8.3.0"

from ultralytics import YOLO
best = "/content/drive/MyDrive/raasta/weights/m3_signs_v1/weights/best.pt"
model = YOLO(best)
for i, n in model.names.items():
    print(i, "->", n)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 8.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
0 -> -
1 -> - 50- probability of horizontal flip
2 -> - 50- probability of vertical flip
3 -> - Random Gaussian blur of between 0 and 3.25 pixels
4 -> - Randomly crop between 0 and 33 percent of the image
5 -> - annotate- and create datasets
6 -> - collaborate with your team on computer vision projects
7 -> - collect - organize images
8 -> -

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="your-api-key")
project = rf.workspace("internship-tvkfo").project("road-signs-and-traffic-lights-detection-and-color-recognition-using-yolov8")
version = project.version(4)
dataset = version.download("yolov8")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 373.9/373.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Road-Signs-and-Traffic-Lights-Detection-and-Color-Recognition-using-YOLOv8-4 in yolov8:: 100%|██████████| 9906/9906 [00:01<00:00, 6586.09it/s]


In [ ]:
import yaml
names = yaml.safe_load(open(f"{dataset.location}/data.yaml"))["names"]
for i, n in enumerate(names):
    print(i, "->", n)

0 -> 20_mph
1 -> Give_way_warn
2 -> bus_stop
3 -> cycle_mark
4 -> do_not_enter
5 -> do_not_stop
6 -> do_not_turn_l
7 -> do_not_turn_r
8 -> do_not_u_turn
9 -> enter_left_lane
10 -> green_light
11 -> left_right_lane
12 -> no_parking
13 -> parking
14 -> ped_crossing
15 -> ped_zebra_cross
16 -> railway_crossing
17 -> red_light
18 -> stop
19 -> t_intersection_l
20 -> traffic_light
21 -> u_turn
22 -> warning
23 -> yellow_light


In [ ]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

from google.colab import drive
drive.mount("/content/drive")

!pip -q install -U "ultralytics>=8.3.0" roboflow

True Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="your-key")
project = rf.workspace("internship-tvkfo").project("road-signs-and-traffic-lights-detection-and-color-recognition-using-yolov8")
dataset = project.version(1).download("yolov8")
print(dataset.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Road-Signs-and-Traffic-Lights-Detection-and-Color-Recognition-using-YOLOv8-1 in yolov8:: 100%|██████████| 6992/6992 [00:02<00:00, 2566.17it/s] 

/content/Road-Signs-and-Traffic-Lights-Detection-and-Color-Recognition-using-YOLOv8-1


In [ ]:
import yaml
names = yaml.safe_load(open(f"{dataset.location}/data.yaml"))["names"]
print(len(names), "classes")
for i, n in enumerate(names):
    print(i, "->", n)

22 classes
0 -> 20_mph
1 -> bus_stop
2 -> do_not_enter
3 -> do_not_stop
4 -> do_not_turn_l
5 -> do_not_turn_r
6 -> do_not_u_turn
7 -> enter_left_lane
8 -> green_light
9 -> left_right_lane
10 -> no_parking
11 -> parking
12 -> ped_crossing
13 -> ped_zebra_cross
14 -> railway_crossing
15 -> red_light
16 -> stop
17 -> t_intersection_l
18 -> traffic_light
19 -> u_turn
20 -> warning
21 -> yellow_light


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=60,
    imgsz=320,
    batch=32,
    patience=15,
    project="/content/drive/MyDrive/raasta/weights",
    name="m3_signs_v2",
    exist_ok=True,
)

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Road-Signs-and-Traffic-Lights-Detection-and-Color-Recognition-using-YOLOv8-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  2,  3,  4,  5,  6,  7,  8,  9, 10, 13, 14, 15, 16, 17, 18, 20, 21])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b7d3811cc20>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    

In [ ]:
best = "/content/drive/MyDrive/raasta/weights/m3_signs_v2/weights/best.pt"
metrics = YOLO(best).val(data=f"{dataset.location}/data.yaml", imgsz=320)
print("mAP50:", metrics.box.map50)

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,137,810 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1303.4±539.7 MB/s, size: 58.0 KB)
val: Scanning /content/Road-Signs-and-Traffic-Lights-Detection-and-Color-Recognition-using-YOLOv8-1/valid/labels.cache... 490 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 490/490 93.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 31/31 6.1it/s 5.1s
                   all        490        531      0.947      0.917      0.959      0.798
                20_mph          2          2      0.858          1      0.995      0.945
          do_not_enter         30         30      0.961          1      0.995      0.891
           do_not_stop         30         30      0.883        0.9      0.949       0.88
         do_not_turn_l         30         34      0.997          1  

In [ ]:
path = YOLO(best).export(format="tflite", imgsz=320)
print(path)
!cp "{path}" /content/drive/MyDrive/raasta/weights/raasta_signs.tflite
!ls -lh /content/drive/MyDrive/raasta/weights/raasta_signs.tflite

WARNING ⚠️ format='tflite' is deprecated as of 8.4.83 and has been replaced by the unified Google LiteRT format. Exporting format='litert' instead. See https://docs.ultralytics.com/integrations/litert
Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 73 layers, 3,137,810 parameters, 0 gradients, 2.2 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/raasta/weights/m3_signs_v2/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 26, 2100) (6.2 MB)
requirements: Ultralytics requirements ['litert-torch>=0.9.0', 'ai-edge-litert>=2.1.4'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 86 packages in 707ms
Prepared 16 packages in 1.58s
Uninstalled 4 packages in 43ms
Installed 16 packages in 38ms
 + ai-edge-litert==2.2.0
 + ai

invalid escape sequence '\.'



LiteRT: starting export with litert_torch 0.9.4...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

...

(00:13) [ DONE] LiteRT-Torch Convert > Run LiteRT Converter Passes (+00:00)

(00:13) [ DONE] LiteRT-Torch Convert (+00:13)

(00:00) [START] Write Model to /content/drive/MyDrive/raasta/weights/m3_signs_v2/weights/best.tflite

(00:00) [ DONE] Write Model to /content/drive/MyDrive/raasta/weights/m3_signs_v2/weights/best.tflite (+00:00)

LiteRT: export success ✅ 19.9s, saved as '/content/drive/MyDrive/raasta/weights/m3_signs_v2/weights/best.tflite' (12.1 MB)

Export complete (20.2s)
Results saved to /content/drive/MyDrive/raasta/weights/m3_signs_v2/weights/best.tflite
Predict:         yolo predict task=detect model=/content/drive/MyDrive/raasta/weights/m3_signs_v2/weights/best.tflite imgsz=320 
Validate:        yolo val task=detect model=/content/drive/MyDrive/raasta/weights/m3_signs_v2/weights/best.tflite imgsz=320 data=/content/Road-Signs-and-Traffic-Lights-Detection-and-Color-Recognition-using-YOLOv8-1/data.yaml  
Visualize:       https://netron.app
/content/drive/MyDrive/raasta/weights/m3_signs_v2/weights/best.tflite
-rw------- 1 root root 13M Oct 11 18:12 /content/drive/MyDrive/raasta/weights/raasta_signs.tflite
